# Imaging configuration checks
A small example of the shared, read-only configuration validators. No data or cluster is required.

In [ ]:
from astroviper.processing_functions.imaging.config_check import (
    CONTINUUM_SINGLE_FIELD,
    ImagingMode,
    check_config,
)

config = {
    "image_params": {
        "frequency_coords": [1.0e9, 1.1e9],
        "reference_frequency": 1.05e9,
        "n_taylor_terms": 2,
    },
    "deconvolver": "mtmfs",
    "continuum_mode": "mfs",
    "cycle_control_params": {"threshold": 0.0},
}
check_config(config, mode=CONTINUUM_SINGLE_FIELD)

Invalid combinations raise without substituting a different algorithm. Both multifield modes are reserved explicitly. At a direct imaging entry point leave `input_checked=False`; only a caller that has validated the unchanged configuration in the same imaging mode should pass `True`. Runtime product checks remain active.

In [ ]:
invalid = {**config, "deconvolver": "hogbom"}
try:
    check_config(invalid, mode=CONTINUUM_SINGLE_FIELD)
except ValueError as error:
    assert "n_taylor_terms=1" in str(error)
else:
    raise AssertionError("Invalid configuration was accepted")
assert invalid["deconvolver"] == "hogbom"

for imaging_kind in ("cube", "continuum"):
    try:
        check_config({}, mode=ImagingMode(imaging_kind, "multifield"))
    except NotImplementedError:
        pass
    else:
        raise AssertionError("Multifield execution is not implemented")

## Spectral sampling after flags and weights

Distributed drivers gather actual effective weights before creating outputs or
graphs. This pure numerical helper shows the resulting rank and condition metrics.
Zero-weight channels contribute no constraints. The spectral metric uses the
2-norm; the reciprocal metric is stored separately.

In [ ]:
from astroviper.processing_functions.imaging.config_check.param_check_residual_update import (
    check_spectral_sampling,
)

spectral = check_spectral_sampling(
    [1.0e9, 1.1e9, 1.2e9],
    [1.0, 0.0, 2.0],
    n_taylor_terms=2,
    reference_frequency=1.05e9,
    context={
        "field": "target",
        "spectral_window": "spw0",
        "channel_selection": [0, 1, 2],
        "polarization": ["I"],
    },
)
assert spectral["usable_channel_count"] == 2
assert spectral["spectral_design_rank"] == 2
spectral

## Derived Taylor products

Product validation always runs, including when configuration was checked by an
upper layer. Every time/polarization plane must pass before any CLEAN update.
Here the two-term PSF moments form a positive definite Hessian. The recorded
Taylor-Hessian metric is the native infinity-norm estimate, separate from the
spectral SVD metric.

In [ ]:
import numpy as np
import xarray as xr

from astroviper.processing_functions.imaging.config_check.param_check_model_update import (
    check_model_products,
)

psf = np.zeros((1, 3, 1, 8, 8), dtype=np.float64)
psf[0, 0, 0, 4, 4] = 1.0
psf[0, 2, 0, 4, 4] = 0.25
image = xr.Dataset(
    {
        "SKY_RESIDUAL": (
            ("time", "taylor_term", "polarization", "l", "m"),
            np.ones((1, 2, 1, 8, 8)),
        ),
        "POINT_SPREAD_FUNCTION": (
            ("time", "psf_taylor_order", "polarization", "l", "m"),
            psf,
        ),
    },
    attrs={
        "data_groups": {
            "residual": {
                "sky": "SKY_RESIDUAL",
                "point_spread_function": "POINT_SPREAD_FUNCTION",
            }
        }
    },
)
products = check_model_products(image, config, mode=CONTINUUM_SINGLE_FIELD)
assert products["effective_scales"] == [0.0]
assert products["taylor_hessian_condition_number"]
products